In [1]:
from sklearn.feature_extraction import DictVectorizer #对字典数据进行向量化
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
import jieba
import numpy as np
from sklearn.impute import SimpleImputer

特征列中含有字符串。如何做特征提取（当成类别）

In [2]:
dict1=DictVectorizer(sparse=True) #sparse=True 稀疏矩阵 二维矩阵很多都是0的时候，就可以用稀疏存储节约空间。缺点，所有机器学习都要转为四四方方的存储。
data=dict1.fit_transform([{'city':'北京','temperature':100},
                          {"city":"上海","temperature":60},
                          {"city":"深圳","temperature":70}])
#区分fit_transform和transform
# transform用于测试集 fit_transform用于训练集
# print(data)
print('-'*50)
print(dict1.get_feature_names_out()) #特征列
print('-'*50)
print(dict1.inverse_transform(data)) #逆转回去
# one-hot 独热码

--------------------------------------------------
['city=上海' 'city=北京' 'city=深圳' 'temperature']
--------------------------------------------------
[{'city=北京': np.float64(1.0), 'temperature': np.float64(100.0)}, {'city=上海': np.float64(1.0), 'temperature': np.float64(60.0)}, {'city=深圳': np.float64(1.0), 'temperature': np.float64(70.0)}]


文本处理，不可以用独热码，因为如果使用独热码，会让矩阵变得非常长 ，有多少样本，就有多少特征。

In [3]:
#统计词频，max_df,min_df 小数（0-1），某个词出现的次数/所有文档数量
#min_df=2
#默认会去除单个字母的单词。默认认为这个词对整个样本没有影响。认为其没有语义
vector1=CountVectorizer(min_df=2) #最小的词频是2，只统计大于等于2的，词频等于1的被过滤了，特征越多，训练越慢。
res=vector1.fit_transform(["life is short, i like python life",
                           "life is too long, i dislike python ",
                           "life is short"])
print(vector1.get_feature_names_out()) #特征列名字
print('-'*50)
print(res.toarray())
print('-'*50)
# print(res)

['is' 'life' 'python' 'short']
--------------------------------------------------
[[1 2 1 1]
 [1 1 1 0]
 [1 1 0 1]]
--------------------------------------------------


英文默认空格分隔，中文用jieba分词,词频代表一个文章，模型学习的是规律，如果一个词是“的”没有实际含义，但是每篇文章都有。会让模型误解，他很重要，但是他没有具体的含义。
tf-idf

In [2]:
def tfdifvec():
    """
    中文特征值化，用于计算tfidf值
    tf 词的频率出现的次数
    idf逆文档频率
    log(总文档的数量/该词出现的文档数量)
    平滑处理：训练的时候有3个特征，测试的时候比如样本里没有这个词，总文档数/（测试集没有这个词的话，这里就是0），这里就会出现零分裂，加一个1，只是为了可计算
    训练用的特征和测试集的特征数是一样的。
    :return:
    """
    tf=TfidfVectorizer(smooth_idf=True)# 做平滑处理，
    return  0


无监督学习，把数据进行分类，没有目标值。分类是超参。有监督学习，训练的时候有目标值
机器学习流程== 数据清洗->特征工程->机器学习->模型评估  如果评估达到要求就上线。

特征工程： dictvectorizer 将字典中的字符串数据变为one-hot编码
countvectorizer 把一串文本里每个词的词频。统计整个文本集中出现的词，把词作为特征，每个样本对应的特征的值是词的词频。如果是汉字就先进行jieba分词。再进行countvectorizer

tfidf：相对于countvectorizer的优势，如果用词频来代表一个文本的含义，会导致出现次数很高的连词（然后，接着）权重过大。 idf逆文档频率=log(总文档数量/对应出现的文档的数量)   词频*逆文档频率

fit_transform和transform  训练集使用fit_transform 测试集使用transform


第120节课
机器学习    特征工程    （归一化、标准化、空值处理、降维）

In [7]:
#归一化 每一列特征量纲不一样。选了4个特征是独立同分布。每个特征对最终结果的影响。把每列特征都拉到同一个量纲。
#并不认为某些特征先天性对结果有影响。相关性系数。
# X'=(x-min)/(max-min)
# X''=X'*(mx-ml)+ml   mx=1; ml=-1
# 归一化的缺点，约会对象数据。有一列是金钱。突然有一个马化腾。这一列本来很重要。其他人的金钱都变成0，模型根据分布就会学不到东西。面对异常值的时候，会直接造成这个特征对结果没效果了。适合小样本数据场景。
mm=MinMaxScaler(feature_range=(0,1))#归一化到0-1的区间
data=mm.fit_transform([[90,2,10,40],[60,4,15,45],[75,3,13,46]])#最好有很多的样本
print(data)
out=mm.transform([[90,2,10,400],[60,4,15,45],[75,3,13,46]])  #transform中的每个元素减的最大最小值是训练集fit_transform的值  (90-60)/(90-60)=1,还是拿训练集的尺子来训练的。
print(out)

[[1.         0.         0.         0.        ]
 [0.         1.         1.         0.83333333]
 [0.5        0.5        0.6        1.        ]]
[[ 1.          0.          0.         60.        ]
 [ 0.          1.          1.          0.83333333]
 [ 0.5         0.5         0.6         1.        ]]


标准化 对于异常值数据不敏感


In [ ]:
# 标准化计算公式 (x-均值)/标准差。
#一旦有马化腾这种，是会出现负值的。